In [1]:
import gzip
import http
import io
from typing import Iterable, Optional
from google.auth import credentials as gcredentials
from google.auth.transport import requests
from google.cloud import storage
from google.oauth2 import credentials
import matplotlib
import nibabel as nib
import numpy as np
from requests_toolbelt.multipart import decoder
import subprocess

In [2]:
TOKEN = subprocess.run(['gcloud', 'auth', 'application-default', 'print-access-token'], stdout=subprocess.PIPE).stdout.decode('utf-8').strip('\n')
TOKEN

'ya29.a0AXeO80Tfe5lxzBSzKn60Jjur2xIRKDK3BFOBKeDLCfauKhawFf2scTzNfp78-hV08WAlBrpsm7Hag2aFqqUEoOeRlIURUFTIhWMMkDrNrI9TWDJAHvhIVXuX08WVZGbjJYz58-REE3xtOl78hq-YGfgS4gaH7-0Z9muli3JJaCgYKATISARISFQHGX2MimxdVYEekajZ82UDc-lZlvw0175'

In [4]:
# Get a list of NIfTI files from the GCS bucket:

project_id = 'fleet-space-445215-f7'
location = 'us-east1'

gcs_storage_client = storage.Client(project_id)
gcs_bucket_name = 'ct_hemorrhage'
gcs_bucket = gcs_storage_client.bucket(gcs_bucket_name)

nifti_urls = []

for a in gcs_bucket.list_blobs():
  if a.name.endswith('.gz'):
    nifti_urls.append('gs://ct_hemorrhage/' + a.name)

print('Files to process:')
print(nifti_urls)

/usr/local/lib/python3.11/dist-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Files to process:
['gs://ct_hemorrhage/ID_0237f3c9_ID_40015688b9.nii.gz']


In [5]:
# OPTIONAL
# Download and check the NIfTI file to make sure it can run through CT Foundation:
client = storage.Client(
    project=None, credentials=credentials.Credentials(TOKEN)
)

for a_nifti in nifti_urls:
  print(f'Checking: {a_nifti}')
  blob = storage.Blob.from_string(a_nifti, client=client)
  with blob.open('rb') as f:
    the_bytes = f.read()

  # Unzip the file
  compressed_stream = io.BytesIO(the_bytes)
  with gzip.GzipFile(fileobj=compressed_stream, mode='rb') as decompressed:
    the_bytes = decompressed.read()
  compressed_stream.close()

  # Load and check NIfTI image
  nifti_image = nib.Nifti1Image.from_bytes(io.BytesIO(the_bytes).read())
  reoriented_img = nib.as_closest_canonical(nifti_image)
  reoriented_img = reoriented_img.get_fdata()

  slope, intercept = nifti_image.header.get_slope_inter()

  print(f'Should be (512,512, x) {reoriented_img.shape}')
  print('Min and Max should be in Hounsfield Units')
  # Note: Some scanners set -3024 for outside of the imaging area
  print(f'Min: {np.min(reoriented_img)}')
  print(f'Max: {np.max(reoriented_img)}')

Checking: gs://ct_hemorrhage/ID_0237f3c9_ID_40015688b9.nii.gz
Should be (512,512, x) (512, 512, 28)
Min and Max should be in Hounsfield Units
Min: -2048.0
Max: 3071.0


In [6]:
# @title Python methods to call CT Foundation's API with NIfTI URLs.

from concurrent.futures import ThreadPoolExecutor
import dataclasses
import functools
import json
from typing import Any, Tuple
import google.auth
import google.auth.transport.requests
import numpy as np


@dataclasses.dataclass(eq=False, frozen=True)
class Response:
  """Response from a Vertex Endpoint."""

  status_code: int
  response_json: dict[str, Any] | None  # json_types.JSONObject


class Endpoint:
  """Calling utility for a Vertex Endpoint using default credentials."""

  def __init__(self):
    self._endpoint_url = (
        'https://us-central1-aiplatform.googleapis.com/v1/projects/'
        'hai-cd3-foundations/locations/us-central1/endpoints/300'
    )

  def predict(
      self,
      instances=list[Any],
      parameters: dict[str, Any] | None = None,
      credentials: google.auth.credentials.Credentials | None = None,
  ) -> Response:
    """Calls the Vertex Endpoint with the given instances and parameters."""
    if credentials is None:
      credentials = google.auth.default()[0]
    session = google.auth.transport.requests.AuthorizedSession(
        credentials=credentials
    )
    response = session.post(
        self._endpoint_url + ':predict',
        json=(
            {'instances': instances}
            | ({'parameters': parameters} if parameters is not None else {})
        ),
        headers={
            'Content-Type': 'application/json',
        },
        timeout=400,
    )
    try:
      response_json = response.json()
    except json.JSONDecodeError:
      # Not expected, handling in case server incorrectly returns non-JSON.
      response_json = None
    return Response(
        status_code=response.status_code,
        response_json=response_json,
    )


def call_single_batch(
    caller: Endpoint, credentials, urls: list[str], access_token: str
) -> list[Tuple[np.ndarray | str, str]]:
  """Handles calls for a single batch and returns embeddings."""
  return_data = []
  if not credentials.valid:
    credentials.refresh(google.auth.transport.requests.Request())
  instances = [
      {'gcs_uri': a_url, 'bearer_token': f'{access_token}'} for a_url in urls
  ]
  returns = caller.predict(instances=instances)
  if returns.status_code != 200:
    for a_url in urls:
      return_data.append((f'FAIL STATUS {returns.status_code}', a_url))
    return return_data
  else:
    for i in range(len(returns.response_json['predictions'])):
      if returns.response_json['predictions'][i]['error_response']:
        return_data.append(
            (returns.response_json['predictions'][i]['error_response'], urls[i])
        )
      else:
        embeddings = returns.response_json['predictions'][i][
            'embedding_result'
        ]['embedding']
        return_data.append((embeddings, urls[i]))
    return return_data


def get_ct_embeddings(
    caller: Endpoint,
    credentials,
    urls: list[str],
    access_token: str,
    batch_size: int,
    parallel_size: int,
) -> list[Tuple[np.ndarray | str, str]]:
  """Handles calls and returns for parallel requests.

  Args:
    caller: CT foundation API caller.
    credentials: The credentials for the API.
    urls: List of urls to the NIfTI files in the cloud bucket. This must be of
      length batch_size * parallel_size.
    access_token: Access token for the DICOM store.
    batch_size: The number of volumes to pass in a batch (max 5).
    parallel_size: The number of parallel calls.

  Returns:
    Tuple list of embeddings | errors and the corresponding urls from which
      the embeddings were computed.
  """
  assert batch_size < 6, 'Batch size must be 5 or less.'
  assert (
      len(urls) == batch_size * parallel_size
  ), 'Error in batch, parallel sizes versus requests'

  # Setup up parallel batches
  p_urls = []
  for i in range(parallel_size):
    p_urls.append(urls[i * batch_size : (i + 1) * batch_size])

  # Check for correct sizing
  assert len(p_urls) == parallel_size, 'Error in batch, parallel dimensions'

  call_batch = functools.partial(call_single_batch, caller, credentials)

  # Launch parallel calls
  with ThreadPoolExecutor(max_workers=parallel_size) as executor:
    futures = [
        executor.submit(call_batch, b_urls, access_token) for b_urls in p_urls
    ]
    results = [f.result() for f in futures]
  # Unpack results into a single list
  return_results = []
  for b_result in results:
    for a_result in b_result:
      return_results.append(a_result)
  return return_results

In [8]:
#@title Create token and call the API for the DICOM volume

# Credentials to access the API
credentials = google.auth.default()[0]

# Token to access the DICOMs in the DICOM store
TOKEN = subprocess.run(['gcloud', 'auth', 'application-default', 'print-access-token'], stdout=subprocess.PIPE).stdout.decode('utf-8').strip('\n')

# Call the API with a single call and a batch size of 1
my_embeddings = get_ct_embeddings(
    caller=Endpoint(), credentials=credentials, urls=nifti_urls,
    access_token=TOKEN, batch_size=1, parallel_size=1)
# Total passed urls are 3
print(f'Total return results: {len(my_embeddings)}')
print('Example from first result....')
print(f'Embeddings or error message for the CT: {my_embeddings[0][1]}')
print(my_embeddings[0][0])

/usr/local/lib/python3.11/dist-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/usr/local/lib/python3.11/dist-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Total return results: 1
Example from first result....
Embeddings or error message for the CT: gs://ct_hemorrhage/ID_0237f3c9_ID_40015688b9.nii.gz
[0.2302913814783096, -0.007925261743366718, -1.568096160888672, -0.5546199679374695, -0.9188135862350464, -1.027007818222046, -1.509534001350403, 1.196481943130493, 0.1968782097101212, 0.6661188006401062, -0.3167558312416077, 0.8546710014343262, -0.8810301423072815, 0.0786610022187233, -0.8881055116653442, -0.2583000361919403, 0.9065435528755188, 0.5733822584152222, 0.4871328771114349, -0.09940846264362335, 1.058911561965942, -0.6176945567131042, 1.094477295875549, -0.3151083588600159, -0.3808581233024597, 1.403467655181885, -0.260700911283493, 0.7343255281448364, -1.696839928627014, -0.3205861151218414, 1.955904960632324, -0.8551091551780701, -0.452226459980011, -0.2116678506135941, -1.774693846702576, 0.2382568717002869, -0.2133896350860596, 1.949414253234863, -0.01619016192853451, 1.42678427696228, 1.879261255264282, -0.3193057775497437, 1